In [ ]:
import os
os.environ["NUMBA_NUM_THREADS"] = "64"

import numpy as np
import importlib
import requests
import xarray as xr
import remap_numba_helpers as rnh
import remap_zarr_helpers as rzh

importlib.reload(rnh)
importlib.reload(rzh)

item = requests.get(
    "https://stac2.cloud.dkrz.de/fastapi/collections/eerie-eerie-mpi-m-icon-esm-er-hist-1950-v20240618/items/eerie-eerie-mpi-m-icon-esm-er-hist-1950-v20240618-disk.model-output.icon-esm-er.hist-1950.v20240618.atmos.native.2d_monthly_mean-zarr-kerchunk"
).json()

asset = "dkrz-disk"
data = xr.open_dataset(
    item["assets"][asset]["href"],
    **item["assets"][asset]["xarray:open_kwargs"],
    storage_options=item["assets"][asset].get("xarray:storage_options"),
)
#data
prepared_max, store_paths = rzh.remap_variables_to_all_zoom_stores(
    source_ds=data,
    variables=["pr"],
    zooms=[9],
    dataset_id="eerie-hist-1950-v20240618",
    frequency_tag="P1M",
    stat_tag="mean",
    out_root="/scratch/k/k202181/remap_eerie/eerie-hist-1950-v20240618",
    cache_root="/scratch/k/k202181/remap_eerie",
    gridfile="/pool/data/ICON/grids/public/mpim/0033/icon_grid_0033_R02B08_G.nc",
    rnh_module=rnh,
    cell_chunk=65536,
    pyramid_agg="nanmean",
    min_zoom=0,
    nest=True,
    use_memmap=False,
    show_progress=True,
    verbose=True,
    rebuild_source_cache=False,
    rebuild_target_cache=False,
    rebuild_weights=False,
    resume=True,
    overwrite_variables=True,
    check_conservation_first_step=False,
    consolidate_at_end=True,
)